In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_52634/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Donovan Mitchell,Over,28.5,-137,2025-11-15,2025-11-15T21:25:00Z
1,Underdog,player_points,Donovan Mitchell,Under,28.5,-137,2025-11-15,2025-11-15T21:25:00Z
2,Underdog,player_points,Evan Mobley,Over,19.5,-137,2025-11-15,2025-11-15T21:25:00Z
3,Underdog,player_points,Evan Mobley,Under,19.5,-137,2025-11-15,2025-11-15T21:25:00Z
4,Underdog,player_points,Kentavious Caldwell-Pope,Over,8.5,-137,2025-11-15,2025-11-15T21:25:00Z


### Update projected starting lineups

In [10]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 10 teams with confirmed lineups


### Top EVs for single bets

In [15]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.30, stake=10, 
                             variance_inflation=1.1, 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 79 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Luka Doncic,BetRivers,30.5,23.24,Under,116,1,7.64,76.4,0.659,High
1,Luka Doncic,BetMGM,32.5,23.24,Under,-110,1,6.86,68.6,0.755,High
2,Scottie Barnes,Bovada,21.5,23.20,Over,185,0,6.78,67.8,0.367,High
3,Luka Doncic,FanDuel,32.5,23.24,Under,-111,1,6.71,67.1,0.744,High
4,Luka Doncic,BetOnline.ag,32.5,23.24,Under,-112,1,6.66,66.6,0.746,High


## Top EVs for 2 leg bets

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 59 players...
Processing 53 players with valid predictions...
Generated 1119 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Scottie Barnes,Luka Dončić,18.5,32.5,over,under,1,6.92,0.346,High,High
1,Pat Connaughton,Luka Dončić,4.5,32.5,over,under,1,6.89,0.344,Med,High
2,Donovan Mitchell,Luka Dončić,28.5,32.5,over,under,1,6.81,0.340,High,High
3,Miles Bridges,Luka Dončić,19.5,32.5,over,under,1,6.70,0.335,High,High
4,Isaiah Jackson,Luka Dončić,7.5,32.5,over,under,1,6.34,0.317,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 73 players...
Processing 65 players with valid predictions...
Generated 1683 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,T.J. McConnell,Luka Dončić,10.0,32.5,under,under,1,9.50,0.475,Low,High
1,Scottie Barnes,Luka Dončić,18.5,32.5,over,under,1,7.24,0.362,High,High
2,Pat Connaughton,Luka Dončić,4.5,32.5,over,under,1,6.91,0.346,Med,High
3,Jock Landale,Luka Dončić,7.5,32.5,over,under,1,6.78,0.339,High,High
4,Miles Bridges,Luka Dončić,19.5,32.5,over,under,1,6.74,0.337,High,High


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1, 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 59 players...
Processing 53 players with valid predictions...
Generated 22651 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Donovan Mitchell,Scottie Barnes,Luka Dončić,28.5,18.5,32.5,over,over,under,0,12.01,0.240,High,High,High
1,Cam Spencer,Scottie Barnes,Luka Dončić,6.5,18.5,32.5,over,over,under,0,11.96,0.239,High,High,High
2,Scottie Barnes,Spencer Jones,Luka Dončić,18.5,6.5,32.5,over,under,under,0,11.93,0.239,High,Low,High
3,Cam Spencer,Spencer Jones,Luka Dončić,6.5,6.5,32.5,over,under,under,0,11.91,0.238,High,Low,High
4,Scottie Barnes,Tony Bradley,Luka Dončić,18.5,6.5,32.5,over,over,under,0,11.87,0.237,High,Med,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=100, 
                     variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 73 players...
Processing 65 players with valid predictions...
Generated 42167 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Donovan Mitchell,Scottie Barnes,Luka Dončić,28.5,18.5,32.5,over,over,under,0,12.01,0.240,High,High,High
1,Cam Spencer,Scottie Barnes,Luka Dončić,6.5,18.5,32.5,over,over,under,0,11.96,0.239,High,High,High
2,Scottie Barnes,Spencer Jones,Luka Dončić,18.5,6.5,32.5,over,under,under,0,11.93,0.239,High,Low,High
3,Cam Spencer,Spencer Jones,Luka Dončić,6.5,6.5,32.5,over,under,under,0,11.91,0.238,High,Low,High
4,Scottie Barnes,Tony Bradley,Luka Dončić,18.5,6.5,32.5,over,over,under,0,11.87,0.237,High,Med,High
